[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C03_LLM_Evals_Course/03_prompt_sensitivity/03_prompt_sensitivity.ipynb)

# 03 · 答案抽取与 prompt 敏感性 —— 实验篇

配套讲解：`03_讲解.html`（模块 03 / 9）。本 notebook 用一个真实小模型把讲解里的三类敏感性**亲手测出来**：

1. **实验 1 · 模板敏感性**：3 个语义等价的 prompt 模板 → 同一批题的分数漂移多少？[Sclar 2023]
2. **实验 2 · 选项顺序敏感性**：每题选项循环移位 4 次 → per-question 一致率 + position bias 矩阵 [Zheng 2023; Alzahrani 2024]
3. **实验 3 · 抽取鲁棒性**：8 种真实风格的模型输出 → 你的正则接得住几种？

后接 3 道 ✏️ 练习：`extract_choice` 兜底链、`format_spread`、`circular_accuracy` —— 都是可以直接搬进自家 harness 的部件。

**⚠️ 运行资源**

- 默认加载 `Qwen/Qwen2.5-1.5B-Instruct`：**首次约 3GB 下载**；全程约 70 次生成，CPU 约 10–20 分钟，Apple Silicon (MPS) / GPU 数分钟。
- 嫌慢可把下方 `MODEL_ID` 换成 `Qwen/Qwen2.5-0.5B-Instruct`（~1GB，更快；小模型的敏感性现象更夸张，教学效果反而更好）。
- **无网络 / 无 transformers 也能跑**：模型加载失败时自动回退到内置的**确定性 mock 响应器**——按固定规则模拟一个"偏好字母 B + 偶发不按格式"的模型，所有实验与练习照常运行且结果可复现（加载 cell 会打印当前用的是真模型还是 mock）。

In [ ]:
import re
import numpy as np

# 10 道内嵌多选题：q = 题干，options = 4 个选项，answer = 正确选项的内容（注意：用内容而非字母
# 标识正确答案——这样选项怎么重排，正确性判定都跟着走）
QUESTIONS = [
    {"q": "What is the capital of France?",
     "options": ["Berlin", "Paris", "Madrid", "Rome"], "answer": "Paris"},
    {"q": "What is 7 * 8?",
     "options": ["56", "54", "48", "64"], "answer": "56"},
    {"q": "Which planet is the largest in the Solar System?",
     "options": ["Earth", "Saturn", "Jupiter", "Neptune"], "answer": "Jupiter"},
    {"q": "What is the chemical formula of water?",
     "options": ["CO2", "H2O", "NaCl", "O2"], "answer": "H2O"},
    {"q": "Who wrote the play 'Romeo and Juliet'?",
     "options": ["Charles Dickens", "Mark Twain", "Jane Austen", "William Shakespeare"],
     "answer": "William Shakespeare"},
    {"q": "How many sides does a hexagon have?",
     "options": ["5", "8", "6", "7"], "answer": "6"},
    {"q": "Which gas do plants primarily absorb for photosynthesis?",
     "options": ["Oxygen", "Nitrogen", "Hydrogen", "Carbon dioxide"], "answer": "Carbon dioxide"},
    {"q": "What is the boiling point of water at sea level in Celsius?",
     "options": ["90", "100", "80", "120"], "answer": "100"},
    {"q": "Which ocean is the largest on Earth?",
     "options": ["Pacific", "Atlantic", "Indian", "Arctic"], "answer": "Pacific"},
    {"q": "What is the square root of 144?",
     "options": ["14", "10", "12", "16"], "answer": "12"},
]
LETTERS = ["A", "B", "C", "D"]

def rotate(options, k):
    '''选项循环右移 k 位：rotate([a,b,c,d], 1) -> [d,a,b,c]（原位置 j 的选项移到 (j+k)%4）'''
    k = k % len(options)
    return options[-k:] + options[:-k]

def gold_letter(options, answer):
    '''正确选项在当前排列下挂在哪个字母上'''
    return LETTERS[options.index(answer)]

print(f"共 {len(QUESTIONS)} 道题；初始排列下正确答案的字母分布：",
      [gold_letter(q["options"], q["answer"]) for q in QUESTIONS])

In [ ]:
# ⚠️ 本 cell 下载约 3GB（首次）。CPU 慢可换 0.5B；无网络则自动回退 mock，全流程仍可跑。
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"   # 更快选项: "Qwen/Qwen2.5-0.5B-Instruct"

class MockResponder:
    '''确定性 mock：按固定规则模拟一个「偏好字母 B + 偶发不按格式」的小模型。

    规则（无随机数，完全可复现）：
      1. 在 prompt 里定位题目，并按各选项文本在 prompt 中的出现顺序还原当前排列，
         得到正确答案当前挂的字母 g 及其位置 k_pos；
      2. token bias：若 (题号 + k_pos) % 4 == 1，无脑回 "Answer: B"（不管对错）；
      3. 其余情况答对，但输出格式按 (题号, k_pos, prompt 长度) 确定性轮换：
         约半数是标准 "Answer: X"，其余为小写句式 / markdown 加粗 / 中文 / 先推理后作答 / 拒答
         ——模拟真实 chat 模型的格式不稳定。'''

    def __init__(self, questions):
        self.questions = questions

    def __call__(self, prompt):
        for idx, item in enumerate(self.questions):
            if item["q"] in prompt:
                break
        else:
            return "I cannot find the question."
        tail = prompt[prompt.index(item["q"]) + len(item["q"]):]   # 只在题干之后找选项
        order = sorted(item["options"], key=lambda o: tail.index(o))
        g = LETTERS[order.index(item["answer"])]
        k_pos = LETTERS.index(g)
        if (idx + k_pos) % 4 == 1:                  # 规则 2：token bias，偏好 B
            return "Answer: B"
        roll = (idx * 7 + k_pos * 3 + len(prompt)) % 10   # 规则 3：格式轮换
        if roll <= 4:
            return f"Answer: {g}"
        if roll == 5:
            return f"the answer is ({g.lower()})."
        if roll == 6:
            return f"**{g}**"
        if roll == 7:
            return f"选 {g}"
        if roll == 8:
            other = "A" if g != "A" else "B"
            return f"Option {other} is tempting, but the correct option is {g}. Answer: {g}"
        return "I'm not sure I can determine the answer."

USE_REAL_MODEL = False
try:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    device = ("cuda" if torch.cuda.is_available()
              else "mps" if torch.backends.mps.is_available() else "cpu")
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype="auto").to(device)
    model.eval()
    USE_REAL_MODEL = True
    print(f"✓ 已加载真实模型 {MODEL_ID}（device={device}）")
except Exception as e:
    mock = MockResponder(QUESTIONS)
    print(f"✗ 模型加载失败（{type(e).__name__}）→ 回退到确定性 mock 响应器。")
    print("  mock 按规则模拟『偏好 B + 偶发不按格式』的模型，实验结论的形态与真模型一致且可复现。")

def ask(prompt, max_new_tokens=48):
    '''统一询问接口：真实模型走 chat template + 贪心解码；否则走 mock。返回原始文本。'''
    if not USE_REAL_MODEL:
        return mock(prompt)
    messages = [{"role": "user", "content": prompt}]
    # transformers 5.x：apply_chat_template(return_tensors="pt") 返回 BatchEncoding（无 .shape），
    # 故用 return_dict=True 取 dict，generate(**inputs) 传入，并以 inputs["input_ids"] 计算 prompt 长度。
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True,
                                           return_tensors="pt", return_dict=True).to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
    prompt_len = inputs["input_ids"].shape[1]
    return tokenizer.decode(out[0, prompt_len:], skip_special_tokens=True).strip()

print("--- smoke test ---")
print(ask("What is the capital of France?\nA) Berlin\nB) Paris\nC) Madrid\nD) Rome\nAnswer:"))

## 实验 1 · 模板敏感性：换个"摆法"，分数动几分？

三个**语义完全等价**的模板，只是排版、标点、措辞风格不同 [Sclar 2023 把这类差异称为 spurious features]：

- `minimal`：裸题干 + `A) 选项` + `Answer:`
- `markdown`：Markdown 标题 + 加粗标号 + 明确格式指令
- `qa_style`：经典 `Q:/A:` 风格 + 格式要求

每个模板跑同样 10 道题，用一个**故意写得很严**的基线抽取器（只认 `Answer: X`）评分，
报告各模板 accuracy、unparseable 数与**极差（spread = max − min）**。

> 预计 30 次生成：CPU 1.5B 约 3–8 分钟，GPU/MPS 约 1 分钟，mock 即时。

In [ ]:
TEMPLATES = {
    "minimal":  "{q}\n{opts_plain}\nAnswer:",
    "markdown": "## Question\n{q}\n\n## Options\n{opts_md}\n\nReply with exactly one line: `Answer: <letter>`.",
    "qa_style": "Q: {q}\nChoices:\n{opts_dot}\nPlease respond in the format 'Answer: X'.\nA:",
}

def render(template, q, options):
    '''把题目 + 当前选项排列填进模板（不同模板用不同的选项标号风格）'''
    return template.format(
        q=q,
        opts_plain="\n".join(f"{L}) {o}" for L, o in zip(LETTERS, options)),
        opts_md="\n".join(f"- **{L}**: {o}" for L, o in zip(LETTERS, options)),
        opts_dot="\n".join(f"{L}. {o}" for L, o in zip(LETTERS, options)),
    )

def baseline_extract(text):
    '''基线抽取器：只认严格的 "Answer: X"。故意很严——它的失误正是本模块的教学点。'''
    m = re.search(r"Answer:\s*([A-D])\b", text)
    return m.group(1) if m else None

results_by_template, raw_outputs = {}, {}
for name, tpl in TEMPLATES.items():
    n_correct, n_unparseable, outs = 0, 0, []
    for item in QUESTIONS:
        out = ask(render(tpl, item["q"], item["options"]))
        outs.append(out)
        pred = baseline_extract(out)
        if pred is None:
            n_unparseable += 1
        elif pred == gold_letter(item["options"], item["answer"]):
            n_correct += 1
    results_by_template[name] = n_correct / len(QUESTIONS)
    raw_outputs[name] = outs
    print(f"{name:>9s}: acc = {results_by_template[name]:.2f}   "
          f"unparseable = {n_unparseable}/{len(QUESTIONS)}")

accs = list(results_by_template.values())
print(f"\n模板极差 (max - min) = {max(accs) - min(accs):.2f}")
print("→ 同一模型、同一批题：分数随『语义等价』的模板表层变化而漂移 [Sclar 2023]")
print("→ 单一模板的分数 = 从模板分布中抽样一次的实现，不是『模型能力』本身\n")
print("几个原始输出（注意格式漂移，严格正则接不住的都成了 unparseable）:")
for name in TEMPLATES:
    print(f"  [{name:>9s}] {raw_outputs[name][7]!r}")

## 实验 2 · 选项顺序：循环移位 + position bias 矩阵

固定 `qa_style` 模板，把每道题的选项**循环移位 4 次**（正确答案依次落到 A/B/C/D 上），统计：

1. **普通 accuracy**（40 次回答按次平均）vs **循环一致率**（一道题 4 次移位全对才算对，
   即 CircularEval 思想，见讲解 §3.3——4 选项下纯猜的循环全对概率只有 $(1/4)^4 \approx 0.4\%$）；
2. **position bias 矩阵**：每个移位下模型选了哪个字母。按设计，每题的正确答案在 4 次移位中
   恰好各落在 A/B/C/D 一次，所以**无偏且全对的模型，字母频率应当完全均匀**；
   某一列显著偏高 = token/position bias [Zheng 2023]，这正是排行榜会被"摆法"重排的微观机制 [Alzahrani 2024]。

> 预计 40 次生成：CPU 1.5B 约 4–10 分钟，GPU/MPS 约 1–2 分钟，mock 即时。

In [ ]:
TPL = TEMPLATES["qa_style"]   # 固定模板，只动选项顺序
K = 4

per_question_preds, per_question_golds = [], []
for item in QUESTIONS:
    preds, golds = [], []
    for k in range(K):
        opts_k = rotate(item["options"], k)
        preds.append(baseline_extract(ask(render(TPL, item["q"], opts_k))))
        golds.append(gold_letter(opts_k, item["answer"]))
    per_question_preds.append(preds)
    per_question_golds.append(golds)

plain_acc = np.mean([p == g
                     for ps, gs in zip(per_question_preds, per_question_golds)
                     for p, g in zip(ps, gs)])
circ = np.mean([all(p == g for p, g in zip(ps, gs))
                for ps, gs in zip(per_question_preds, per_question_golds)])
print(f"普通 accuracy（40 次按次平均） = {plain_acc:.2f}")
print(f"循环一致率（4 次全对才算对）   = {circ:.2f}   ← 通常显著更低：蒙对/偏置对的题被挤掉了")

cols = LETTERS + ["None"]
matrix = np.zeros((K, len(cols)), dtype=int)
for ps in per_question_preds:
    for k, p in enumerate(ps):
        matrix[k][LETTERS.index(p) if p in LETTERS else 4] += 1
print("\nposition bias 矩阵（行 = 移位 k，列 = 模型所选字母的计数；None = unparseable）")
print("       " + "  ".join(f"{c:>4s}" for c in cols))
for k in range(K):
    print(f"k={k}:  " + "  ".join(f"{v:>4d}" for v in matrix[k]))

freq = matrix[:, :4].sum(axis=0)
print("\n各字母被选总次数:", ", ".join(f"{L}={f}" for L, f in zip(LETTERS, freq)))
print("（无偏模型应接近均匀；某字母显著偏高即 token bias [Zheng 2023]）")

## 实验 3 · 抽取鲁棒性：8 种真实风格的输出

实验 1/2 里那个只认 `Answer: X` 的基线抽取器漏掉了不少回答。真实评测中模型输出五花八门——
下面是 8 种真实风格（理想格式 / 中文 / 加粗 / 小写带括号 / 先推理后作答 / 拒答 / 复述选项 / 换种句式声明）。
先看朴素抽取（"抓第一个出现的大写字母"）错在哪：它的失误**不是随机的**——
系统性漏掉小写句式，并在"先推理后作答"时抓到第一个被讨论的（错误）字母，
专门惩罚做显式推理的模型（讲解 §4.1）。

## ✏️ 练习 1：实现 `extract_choice(text)`

实现一个鲁棒的答案抽取函数，签名 `extract_choice(text) -> 'A' | 'B' | 'C' | 'D' | None`，
通过下方全部 8 个 assert 用例。

**提示**（兜底链按严→宽排列，命中即返回；15 行左右可完成）：
1. 显式声明：`answer is X` / `Answer: X` / `correct option is X` / `选 X` ——
   大小写不敏感（`re.IGNORECASE`）、容忍括号 `(b)`；命中多处时**取最后一处**
   （用 `re.finditer` 收集 `(m.start(), 字母)` 再取 `max`）；
2. markdown 加粗 `**X**`；
3. 行首 `X)` 风格（模型复述了整个选项）；
4. 都没有 → 返回 `None`（千万不要瞎猜，unparseable 必须显式化）。

In [ ]:
# 8 种真实风格的模型输出 → (文本, 期望抽取结果)
EXTRACT_CASES = [
    ("Answer: B",                                              "B"),
    ("选 B",                                                   "B"),
    ("**B**",                                                  "B"),
    ("the answer is (b).",                                     "B"),
    ("Option A is tempting since Paris is in France, but option C matches the question. Answer: C", "C"),
    ("I'm not sure I can determine this.",                     None),
    ("B) Paris",                                               "B"),
    ("The correct option is D.",                               "D"),
]

def naive_extract(text):
    '''朴素抽取：抓第一个独立出现的大写字母 A–D'''
    m = re.search(r"\b([A-D])\b", text)
    return m.group(1) if m else None

print("naive_extract 成绩单:")
n_ok = 0
for text, expected in EXTRACT_CASES:
    got = naive_extract(text)
    n_ok += (got == expected)
    print(f"  {'✓' if got == expected else '✗'} expected={expected!r:<6} got={got!r:<6} | {text[:58]}")
print(f"→ {n_ok}/8。两类系统性失误：小写句式漏抽；推理在前时抓到第一个被讨论的错误字母。\n")

def extract_choice(text):
    '''从模型自由输出中抽取选项字母，返回 'A'/'B'/'C'/'D' 或 None（无法解析）。
    实现讲解 §4.1 的兜底链：①显式声明(取最后) → ②加粗 → ③行首 X) → ④None'''
    # TODO: 在这里实现你的兜底链
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测：8 个用例全部通过才算过 ——
for text, expected in EXTRACT_CASES:
    got = extract_choice(text)
    assert got == expected, f"输入 {text!r}: 期望 {expected!r}, 得到 {got!r}"
print("✅ 练习 1 通过：8/8 用例全部正确")

## ✏️ 练习 2：实现 `format_spread(results_by_template)`

把实验 1 的"极差"封装成可复用的报告函数：输入 `{模板名: accuracy}` 字典，
返回二元组 `(spread, std)`：

- `spread` = 最大值 − 最小值（FORMATSPREAD 的核心统计量 [Sclar 2023]）；
- `std` = 各模板 accuracy 的**总体标准差**（`ddof=0`，即 `np.std` 默认行为）。

**提示**：3 行以内可完成；注意单模板时两个值都应是 `0.0`（边界用例）。

In [ ]:
def format_spread(results_by_template):
    '''输入 {模板名: accuracy}，返回 (spread, std)。
    spread = max - min；std = 总体标准差（ddof=0）。'''
    # TODO: 提取所有 accuracy，计算极差与标准差
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
import math
s, d = format_spread({"t1": 0.8, "t2": 0.6, "t3": 0.7})
assert math.isclose(s, 0.2, abs_tol=1e-9), f"spread 应为 0.2，得到 {s}"
assert math.isclose(d, math.sqrt((0.1**2 + 0.1**2 + 0.0**2) / 3), abs_tol=1e-9), f"std 错误: {d}"
s, d = format_spread({"only": 0.55})           # 边界：单模板 → 无离散度
assert s == 0.0 and d == 0.0, (s, d)
s, d = format_spread(results_by_template)      # 用在实验 1 的真实结果上
print(f"实验 1：模板极差 = {s:.2f}，模板间 std = {d:.3f}")
print("→ 报告时应写：acc = 均值 ± std（跨模板），而非单点")
print("✅ 练习 2 通过")

## ✏️ 练习 3：实现 `circular_accuracy(preds, golds)`

实现 CircularEval 的核心指标："一道题在**全部**循环移位下都答对，才记这题为对"（讲解 §3.3）：

$$\mathrm{acc}_{\mathrm{circ}} = \frac{1}{n}\sum_{i=1}^{n}\ \prod_{k=0}^{K-1}\mathbf{1}\big[\hat{y}_i^{(k)} = y_i^{(k)}\big]$$

- 输入：`preds` 与 `golds` 均为 `[n 题][K 个移位]` 的嵌套列表（元素是字母；`preds` 里可能有 `None`，
  unparseable 当然算答错）；
- 输出：通过比例，`float ∈ [0, 1]`。

**提示**：`all(...)` + 一个推导式即可，5 行以内；写完先过合成数据 assert，再用到实验 2 的真实数据上。

In [ ]:
def circular_accuracy(preds, golds):
    '''CircularEval：全部移位都答对才算这道题对。
    preds/golds: [n][K] 嵌套列表（preds 可含 None）。返回通过比例。'''
    # TODO: 对每道题判断"是否 K 次全对"，再求比例
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测（合成数据数值已人工算好）——
import math
synth_preds = [["A", "B", "C", "D"],    # 全对
               ["A", "B", "C", "A"],    # 错 1 次 → 该题不通过
               [None, "B", "C", "D"],   # 有 unparseable → 该题不通过
               ["D", "A", "B", "C"]]    # 全对
synth_golds = [["A", "B", "C", "D"],
               ["A", "B", "C", "D"],
               ["A", "B", "C", "D"],
               ["D", "A", "B", "C"]]
assert math.isclose(circular_accuracy(synth_preds, synth_golds), 2 / 4), "应为 0.5"
assert circular_accuracy([["A"]], [["A"]]) == 1.0      # 边界：K=1 退化为普通 acc
assert circular_accuracy([["A"], ["B"]], [["B"], ["A"]]) == 0.0
ca = circular_accuracy(per_question_preds, per_question_golds)   # 实验 2 的真实数据
print(f"实验 2 的 circular accuracy = {ca:.2f}（应与实验 2 打印的循环一致率相同）")
print("✅ 练习 3 通过")

## 📖 参考答案

**先自己做，再对照。** 运行下方 cell 会覆盖你的实现——之后可回到各自测 cell 重新运行验证。

In [ ]:
# ============ 参考答案（先自己做，再对照！）============

# ---------- 练习 1：extract_choice ----------
def extract_choice(text):
    t = text.strip()
    # ① 显式声明（大小写不敏感，容忍括号；多处命中取【最后】一处）
    pats = [r"(?:answer\s+is|answer\s*[:：]|correct\s+option\s+is|correct\s+answer\s+is)\s*\(?([A-Da-d])\)?\b",
            r"选\s*\(?([A-D])\)?",
            r"答案\s*(?:是|为)?\s*[:：]?\s*\(?([A-Da-d])\)?\b"]
    hits = []
    for p in pats:
        hits += [(m.start(), m.group(1).upper()) for m in re.finditer(p, t, re.IGNORECASE)]
    if hits:
        return max(hits)[1]          # 按出现位置取最后一次声明
    # ② markdown 加粗 **X**
    m = re.search(r"\*\*\(?([A-Da-d])\)?\*\*", t)
    if m:
        return m.group(1).upper()
    # ③ 行首 "X)" / "X." / "(X):" 风格（模型复述选项）
    m = re.match(r"\(?([A-Da-d])[).．:：]", t)
    if m:
        return m.group(1).upper()
    # ④ 整段输出就是一个字母
    m = re.fullmatch(r"\(?([A-Da-d])\)?[.。]?", t)
    if m:
        return m.group(1).upper()
    return None                       # unparseable 显式化，绝不瞎猜

# ---------- 练习 2：format_spread ----------
def format_spread(results_by_template):
    accs = list(results_by_template.values())
    return max(accs) - min(accs), float(np.std(accs))

# ---------- 练习 3：circular_accuracy ----------
def circular_accuracy(preds, golds):
    n_pass = sum(all(p == g for p, g in zip(ps, gs)) for ps, gs in zip(preds, golds))
    return n_pass / len(preds)

print("参考实现已加载——回到上面的三个自测 cell 重新运行即可验证。")
print("延伸实验：把实验 1/2 里的 baseline_extract 换成 extract_choice 重跑，")
print("观察 unparseable 率下降、accuracy 上升——这个差值就是讲解 §5 说的『格式/抽取损失』。")

## 小结

- **同一模型、同一批题**：换模板（实验 1）、换选项摆法（实验 2）、换抽取规则（实验 3），
  分数就会漂移——你在本机复现了 [Sclar 2023; Alzahrani 2024; Zheng 2023] 的核心现象。
- 三个可复用部件：`extract_choice`（兜底链 + 显式 None）、`format_spread`（多模板报极差与 std）、
  `circular_accuracy`（把鲁棒性乘进正确性定义）——可直接搬进你自己的 harness。
- 报告规范回顾（讲解 §7）：多模板均值±方差、固定并公开 harness 全配置、报告 unparseable 率。

**下一站 · 模块 04（`../04_llm_judge/04_讲解.html`）**：当答案根本没有标准字母可抽
——开放生成、写作、多轮对话——就要请 LLM 来当裁判。而裁判自己也带着 position bias、
冗长偏好与自我偏好，敏感性问题换了个形态继续存在。

---
## 🎯 真实数据胶囊题：真实答案在多种格式下的鲁棒抽取

同一个正确答案，模型可能写成“42”、“$42”、“The answer is 42.”、“42.0”。脆弱的抽取器会把对的判成错。用真实 GSM8K 金标答案造多种格式，实现鲁棒抽取器，验证抽取一致。

> 本模块新增的**真实数据**练习：自包含、用真实公开数据把本章方法跑一遍。先做 TODO，`assert` 全过即通关，文末有参考答案。

In [ ]:
import os, json, urllib.request, re
import numpy as np
CACHE=os.path.expanduser("~/.llm_evals_data"); os.makedirs(CACHE,exist_ok=True)
def _f(url,fn):
    p=os.path.join(CACHE,fn)
    if not os.path.exists(p): urllib.request.urlretrieve(url,p)
    return p
def gsm8k(n=300):
    p=_f("https://raw.githubusercontent.com/openai/grade-school-math/master/grade_school_math/data/test.jsonl","gsm8k_test.jsonl")
    return [json.loads(l) for l in open(p).read().splitlines()[:n]]
def gold(ans): return ans.split("####")[-1].strip().replace(",","")
def shakespeare():
    return open(_f("https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt","shake.txt")).read()

rows=gsm8k(100)
golds=[gold(r["answer"]) for r in rows]
def formats(g):  # 同一答案的多种真实书写
    return [g, f"${g}", f"The answer is {g}.", f"#### {g}", f" {g} ", f"{g}.00"]
print("示例金标:", golds[0], "-> 格式:", formats(golds[0]))

**练习**：实现 `extract_number(text)`：从任意上述格式里抽出规范化数字串（去 `$`、去尾随 `.00`、去空白）。目标：同一答案的所有格式都抽出相同结果。

In [ ]:
def extract_number(text):
    # TODO: 去掉 $ 和文字，取最后一个数字，规范化(去 .0/.00 等尾零)
    raise NotImplementedError


In [ ]:
# 自测：所有格式应抽出同一个规范化答案
ok=0
for g in golds:
    outs={extract_number(f) for f in formats(g)}
    if len(outs)==1 and outs.pop()==g.lstrip("0") or True:
        # 规范化后所有格式一致
        outs={extract_number(f) for f in formats(g)}
        if len(outs)==1: ok+=1
acc=ok/len(golds)
assert acc>0.9, f"鲁棒抽取应让>90%题目所有格式一致, 得到{acc:.2f}"
print(f"鲁棒抽取: {acc:.0%} 的题目所有格式抽出一致结果 ✓")


### 📖 参考答案

In [ ]:
def extract_number(text):
    t=text.replace("$","").replace(",","")
    nums=re.findall(r"-?\d+\.?\d*", t)
    if not nums: return None
    x=nums[-1]
    if "." in x: x=x.rstrip("0").rstrip(".")   # 42.00 -> 42
    return x
print("✓ prompt/格式敏感性常被低估，鲁棒抽取器能避免把对的判成错")